In [ ]:
import requests
import pandas as pd
import os
import json
from dotenv import load_dotenv
import matplotlib.pyplot as plt

In [3]:
load_dotenv()
EPI_API_KEY = os.getenv("EPI_API_KEY")

In [4]:
# API call to database

meta_url = "https://api.eia.gov/v2/electricity/electric-power-operational-data"
params = {"api_key": EPI_API_KEY}

r = requests.get(meta_url, params=params) # 'get' visits the website, returns JSON
print(r.status_code) # printing the status code, 200 = success

meta = r.json() 

print(json.dumps(meta, indent=2)[:4000]) # prints the json so it is readable

200
{
  "response": {
    "id": "electric-power-operational-data",
    "name": "Electric Power Operations (Annual and Monthly)",
    "description": "Monthly and annual electric power operations by state, sector, and energy source.\n    Source: Form EIA-923",
    "frequency": [
      {
        "id": "monthly",
        "description": "One data point for each month.",
        "query": "M",
        "format": "YYYY-MM"
      },
      {
        "id": "quarterly",
        "description": "One data point every 3 months.",
        "query": "Q",
        "format": "YYYY-\"Q\"Q"
      },
      {
        "id": "annual",
        "description": "One data point for each calendar year.",
        "query": "A",
        "format": "YYYY"
      }
    ],
    "facets": [
      {
        "id": "location",
        "description": "State / Census Region"
      },
      {
        "id": "sectorid",
        "description": "Sector"
      },
      {
        "id": "fueltypeid",
        "description": "Energy Source"
   

In [5]:
for f in meta["response"]["frequency"]:
    print(f["id"], f["query"])

monthly M
quarterly Q
annual A


In [6]:
# pull real data
meta_url1 = "https://api.eia.gov/v2/electricity/electric-power-operational-data/data/"

params1 = {'frequency':'annual',
           'data[]':'generation',
           'facets[location][]':'US',
        #    'facets[fueltypeid][]':'',
           'start':'2001',
           'end':'2026',
           'sort[0][column]':'period',
            'sort[0][direction]':'asc',
            'api_key': EPI_API_KEY
           }

r = requests.get(meta_url1, params=params1) # 'get' visits the website, returns JSON
print(r.status_code) # printing the status code, 200 = success

meta = r.json() 



200


In [7]:
print("Total records:", meta["response"]["total"])
print(json.dumps(meta["response"]["data"][:3], indent=2))

Total records: 11919
[
  {
    "period": "2001",
    "location": "US",
    "stateDescription": "U.S. Total",
    "sectorid": "1",
    "sectorDescription": "Electric Utility",
    "fueltypeid": "ALL",
    "fuelTypeDescription": "all fuels",
    "generation": "2629945.673",
    "generation-units": "thousand megawatthours"
  },
  {
    "period": "2001",
    "location": "US",
    "stateDescription": "U.S. Total",
    "sectorid": "1",
    "sectorDescription": "Electric Utility",
    "fueltypeid": "AOR",
    "fuelTypeDescription": "all renewables",
    "generation": "1665.73868",
    "generation-units": "thousand megawatthours"
  },
  {
    "period": "2001",
    "location": "US",
    "stateDescription": "U.S. Total",
    "sectorid": "1",
    "sectorDescription": "Electric Utility",
    "fueltypeid": "BIO",
    "fuelTypeDescription": "biomass",
    "generation": "1374.98968",
    "generation-units": "thousand megawatthours"
  }
]


In [ ]:
eia_df = pd.DataFrame(meta["response"]["data"])
eia_df.head()

,period,location,stateDescription,sectorid,sectorDescription,fueltypeid,fuelTypeDescription,generation,generation-units
0,2001,US,U.S. Total,1,Electric Utility,ALL,all fuels,2629945.673,thousand megawatthours
1,2001,US,U.S. Total,1,Electric Utility,AOR,all renewables,1665.73868,thousand megawatthours
2,2001,US,U.S. Total,1,Electric Utility,BIO,biomass,1374.98968,thousand megawatthours
3,2001,US,U.S. Total,1,Electric Utility,BIT,bituminous coal,None,thousand megawatthours
4,2001,US,U.S. Total,1,Electric Utility,COL,"coal, excluding waste coal",1560145.542,thousand megawatthours


In [17]:
# seeing the data
print(f'Periods:{eia_df['period'].unique()}')
print(f'Locations: {eia_df['location'].unique()}')
print(f'State Descriptions: {eia_df['stateDescription'].unique()}')
print(f'Sector ID:  {eia_df['sectorid'].unique()}')
print(f'Sector Descriptions:  {eia_df['sectorDescription'].unique()}')
print(f'Fuel Type ID:  {eia_df['fueltypeid'].unique()}')
print(f'Fuel Type Description:  {eia_df['fuelTypeDescription'].unique()}')


Periods:['2001' '2002' '2003' '2004' '2005' '2006' '2007' '2008' '2009' '2010'
 '2011' '2012']
Locations: ['US']
State Descriptions: ['U.S. Total']
Sector ID:  ['1' '2' '3' '4' '5' '6' '7' '90' '94' '95' '96' '97' '98' '99']
Sector Descriptions:  ['Electric Utility' 'IPP Non-CHP' 'IPP CHP' 'Commercial Non-CHP'
 'Commercial CHP' 'Industrial Non-CHP' 'Industrial CHP'
 'Electric Power Sector Non-CHP' 'Independent Power Producers'
 'Coal Consumption' 'All Commercial' 'All Industrial' 'Electric Power'
 'All Sectors']
Fuel Type ID:  ['ALL' 'AOR' 'BIO' 'BIT' 'COL' 'COW' 'DFO' 'FOS' 'GEO' 'HPS' 'HYC' 'LIG'
 'MLG' 'NG' 'NGO' 'NUC' 'OOG' 'ORW' 'OTH' 'PC' 'PEL' 'PET' 'RFO' 'SUB'
 'SUN' 'WAS' 'WND' 'WOC' 'WOO' 'WWW' 'BIS' 'LFG' 'MSB' 'OB2' 'OBW' 'REN'
 'SPV' 'WNT' 'STH' 'ANT' 'RC']
Fuel Type Description:  ['all fuels' 'all renewables' 'biomass' 'bituminous coal'
 'coal, excluding waste coal' 'all coal products' 'distillate fuel oil'
 'fossil fuels' 'geothermal' 'hydro-electric pumped storage'
 'co